## weighted self attention

In [3]:
import torch

input = torch.tensor([[0.72, 0.45, 0.31], # Dream 
                    [0.75, 0.20, 0.55], # big 
                    [0.30, 0.80, 0.40], # and 
                    [0.85, 0.35, 0.60], # work 
                    [0.55, 0.15, 0.75], # for 
                    [0.25, 0.20, 0.85]]) # it 
words = ['Dream','big','and','work','for','it']

In [8]:
x_2 = input[1]
d_in = input.shape[1]
d_out = 2
print(x_2)
print(d_in)
print(d_out)

tensor([0.7500, 0.2000, 0.5500])
3
2


#### randomly initializing Wk,Wq,Wv

In [9]:
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad=False)

In [10]:
print(W_query,W_key,W_value)

Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]]) Parameter containing:
tensor([[0.1366, 0.1025],
        [0.1841, 0.7264],
        [0.3153, 0.6871]]) Parameter containing:
tensor([[0.0756, 0.1966],
        [0.3164, 0.4017],
        [0.1186, 0.8274]])


### calculation Q,K,V using Wq,Wk,Wv

In [12]:
Q = input @ W_query
K = input @ W_key
V = input @ W_value

In [15]:
attention_scores = Q @ K.T
print(attention_scores)

tensor([[0.6807, 0.6795, 0.9526, 0.8454, 0.7654, 0.8359],
        [0.7021, 0.6990, 0.9867, 0.8707, 0.7880, 0.8624],
        [0.7350, 0.7315, 1.0337, 0.9113, 0.8248, 0.9029],
        [0.8436, 0.8402, 1.1848, 1.0464, 0.9471, 1.0361],
        [0.7080, 0.7025, 1.0003, 0.8764, 0.7929, 0.8699],
        [0.6680, 0.6606, 0.9486, 0.8254, 0.7465, 0.8210]])


In [18]:
from math import sqrt
d_k = K.shape[-1] # for getting dimension of matrix K
print(d_k)
scaled_attention_score = attention_scores/sqrt(d_k)
print(scaled_attention_score)

2
tensor([[0.4813, 0.4805, 0.6736, 0.5978, 0.5412, 0.5911],
        [0.4964, 0.4943, 0.6977, 0.6157, 0.5572, 0.6098],
        [0.5198, 0.5172, 0.7310, 0.6444, 0.5832, 0.6384],
        [0.5965, 0.5941, 0.8378, 0.7399, 0.6697, 0.7327],
        [0.5006, 0.4967, 0.7073, 0.6197, 0.5607, 0.6151],
        [0.4723, 0.4671, 0.6708, 0.5836, 0.5278, 0.5805]])


In [19]:
attention_weights = torch.softmax(scaled_attention_score,dim = -1)
print(attention_weights)

tensor([[0.1536, 0.1534, 0.1861, 0.1725, 0.1630, 0.1714],
        [0.1531, 0.1528, 0.1873, 0.1725, 0.1627, 0.1715],
        [0.1525, 0.1521, 0.1884, 0.1728, 0.1625, 0.1717],
        [0.1505, 0.1501, 0.1915, 0.1737, 0.1619, 0.1724],
        [0.1530, 0.1524, 0.1881, 0.1724, 0.1625, 0.1716],
        [0.1538, 0.1530, 0.1875, 0.1719, 0.1625, 0.1713]])


In [23]:
row_sums = attention_weights.sum(dim=1)

print(row_sums)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [ ]:
col_sums = attention_weights.sum(dim=1)

print(row_sums)

### output_context_vector 

In [24]:
# output_context_vector = attention_weights * transformed_input

output = attention_weights @ V
print(output)

tensor([[0.2273, 0.7361],
        [0.2274, 0.7362],
        [0.2276, 0.7363],
        [0.2280, 0.7368],
        [0.2275, 0.7362],
        [0.2275, 0.7360]])


### same inside a single python class

In [30]:
import torch.nn as nn
from math import sqrt
class Self_attention(nn.Module):
    def __init__(self,d_in,d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in,d_out))
        self.W_key = nn.Parameter(torch.rand(d_in,d_out))
        self.W_value = nn.Parameter(torch.rand(d_in,d_out))
    
    def forward(self,x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value

        att_scores = queries @ keys.T
        att_weights = torch.softmax(att_scores/sqrt(keys.shape[-1]),dim = -1)

        context_vec = att_weights @ values 
        return context_vec



In [31]:
torch.manual_seed(123)
sa_v1 = Self_attention(d_in = input.shape[-1], d_out = 2)
output = sa_v1(input)
print(output)

tensor([[0.2273, 0.7361],
        [0.2274, 0.7362],
        [0.2276, 0.7363],
        [0.2280, 0.7368],
        [0.2275, 0.7362],
        [0.2275, 0.7360]], grad_fn=<MmBackward0>)
